In [2]:
import os
import tensorflow as tf

from models.autoencoder import unet_autoencoder
from models.losses import Loss
from models.data_loader import data_generator
from utilities import gather_image_from_dir

In [3]:
# Data
image_width = 320
image_height = 320
image_channels = 1

# train
train_images_dir = '../data/training/images/'
train_labels_dir = '../data/training/masks/'
# test
test_images_dir = '../data/testing/images/'
test_labels_dir = '../data/testing/masks/'

# Directory for weight saving (creates if it does not exist)
weights_output_dir = 'weights_output/'
weights_output_name = 'UNet4_res_assp_5x5_16k_64x64'
# batch size. How many samples you want to feed in one iteration?
batch_size = 2
# number_of_epoch. How many epochs you want to train?
number_of_epoch = 100
# After how many epochs you want to reduce learning rate by half?
lr_scheduling_epochs = 35
# initial learning rate
initial_lr = 0.001


In [16]:
# Generate image ROI

from processing import split_image_to_tiles
roi = split_image_to_tiles(4288, 2848, 128, 128, 64, 32);

In [18]:
train_images = gather_image_from_dir(train_images_dir)
train_labels = gather_image_from_dir(train_labels_dir)

In [4]:
class CustomSaver(tf.keras.callbacks.Callback):
    def __init__(self):
        self.best_val_score = 0.0

    def on_epoch_end(self, epoch, logs=None):
        # also save if validation error is smallest
        if 'val_dice_eval' in logs.keys():
            val_score = logs['val_dice_eval']
            if val_score > self.best_val_score:
                self.best_val_score = val_score
                print('New best weights found!')
                self.model.save(weights_output_dir + 'best_weights.hdf5')
        else:
            print('Key val_dice_eval does not exist!')


# This function keeps the learning rate at 0.001 for the first ten epochs
# and decreases it exponentially after that.
def scheduler(epoch):
    step = epoch // lr_scheduling_epochs
    lr = initial_lr / 2 ** step
    print('Epoch: ' + str(epoch) + ', learning rate = ' + str(lr))
    return lr

In [5]:
# check how many train and test samples are in the directories
train_images_count = len(gather_image_from_dir(train_images_dir))
train_labels_count = len(gather_image_from_dir(train_labels_dir))
train_samples_count = min(train_images_count, train_labels_count)
print('Training samples: ' + str(train_samples_count))

test_images_count = len(gather_image_from_dir(test_images_dir))
test_labels_count = len(gather_image_from_dir(test_labels_dir))
test_samples_count = min(test_images_count, test_labels_count)
print('Testing samples: ' + str(test_samples_count))

# how many iterations in one epoch? Should cover whole dataset. Divide number of data samples from batch size
number_of_train_iterations = train_samples_count // batch_size
number_of_test_iterations = test_samples_count // batch_size

Training samples: 53
Testing samples: 27


In [6]:
    # Define model
model = unet_autoencoder(filters_in_input=16,
                            input_size=(image_width, image_width, image_channels),
                            loss_function=Loss.CROSSENTROPY50DICE50,
                            downscale_times=4,
                            learning_rate=1e-3,
                            use_se=True,
                            use_aspp=True,
                            use_coord_conv=False,
                            use_residual_connections=True,
                            leaky_relu_alpha=0.1)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 64, 64, 1)]  0           []                               
                                                                                                  
 conv2d (Conv2D)                (None, 64, 64, 16)   400         ['input_1[0][0]']                
                                                                                                  
 batch_normalization (BatchNorm  (None, 64, 64, 16)  64          ['conv2d[0][0]']                 
 alization)                                                                                       
                                                                                                  
 leaky_re_lu (LeakyReLU)        (None, 64, 64, 16)   0           ['batch_normalization[0][0]']

In [8]:


# Define data generator that will take images from directory
train_data_generator = data_generator(batch_size,
                                    image_folder=train_images_dir,
                                    label_folder=train_labels_dir,
                                    target_size=(image_width, image_height),
                                    image_color_mode='grayscale')

test_data_generator = data_generator(batch_size,
                                    image_folder=test_images_dir,
                                    label_folder=test_labels_dir,
                                    target_size=(image_width, image_height),
                                    image_color_mode='grayscale')

train_data_generator

<generator object data_generator at 0x0000015C23CFF7D0>

In [9]:
# create weights output directory
if not os.path.exists(weights_output_dir):
    print('Output directory doesnt exist!\n')
    print('It will be created!\n')
    os.makedirs(weights_output_dir)

# Define template of each epoch weight name. They will be save in separate files
weights_name = weights_output_dir + weights_output_name + "-{epoch:03d}-{loss:.4f}.hdf5"
# Custom saving for the best-performing weights
saver = CustomSaver()
# Learning rate scheduler
learning_rate_scheduler = tf.keras.callbacks.LearningRateScheduler(scheduler)
# Make checkpoint for saving each
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(weights_name, monitor='loss', verbose=1, save_best_only=False,
                                                    save_weights_only=False)

In [10]:


model.fit(train_data_generator,
        steps_per_epoch=number_of_train_iterations,
        epochs=number_of_epoch,
        validation_data=test_data_generator,
        validation_steps=number_of_test_iterations,
        callbacks=[model_checkpoint, learning_rate_scheduler, saver],
        shuffle=True)

Found 53 images belonging to 1 classes.
Found 53 images belonging to 1 classes.
Epoch: 0, learning rate = 0.001
Epoch 1/100
26/26 [==============================] - ETA: 0s - loss: 0.7373 - dice_eval: 0.0196Found 27 images belonging to 1 classes.
Found 27 images belonging to 1 classes.

Epoch 1: saving model to weights_output\UNet4_res_assp_5x5_16k_64x64-001-0.7373.hdf5
New best weights found!
26/26 [==============================] - 16s 318ms/step - loss: 0.7373 - dice_eval: 0.0196 - val_loss: 0.8122 - val_dice_eval: 0.0103 - lr: 0.0010
Epoch: 1, learning rate = 0.001
Epoch 2/100
26/26 [==============================] - ETA: 0s - loss: 0.7116 - dice_eval: 0.0086
Epoch 2: saving model to weights_output\UNet4_res_assp_5x5_16k_64x64-002-0.7116.hdf5
26/26 [==============================] - 8s 262ms/step - loss: 0.7116 - dice_eval: 0.0086 - val_loss: 0.7852 - val_dice_eval: 0.0052 - lr: 0.0010
Epoch: 2, learning rate = 0.001
Epoch 3/100
26/26 [==============================] - ETA: 0s - lo

In [14]:
# Predict
import cv2
from processing import tensor_to_image, image_to_tensor
from utilities import gather_image_from_dir

image_paths = gather_image_from_dir(test_images_dir)

for image_path in image_paths:
    # Load image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    # preprocess
    norm_image = image_to_tensor(image)
    # predict
    prediction = model.predict(norm_image)
    # make image uint8
    prediction_image = tensor_to_image(prediction)

    # Do you want to visualize image?
    show_image = True
    if show_image:
        cv2.imshow("image", image)
        cv2.imshow("prediction", prediction_image)
        cv2.waitKey(1000)

ValueError: in user code:

    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 1801, in predict_function  *
        return step_function(self, iterator)
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 1790, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 1783, in run_step  **
        outputs = model.predict_step(data)
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 1751, in predict_step
        return self(x, training=False)
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\utils\traceback_utils.py", line 67, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\input_spec.py", line 264, in assert_input_compatibility
        raise ValueError(f'Input {